# Load origin data

In [1]:
import pandas as pd
import numpy as np

In [2]:
import os

origin_data_dir = "data/origin_data"
transactions_file = "credit_card_transactions-ibm_v2.csv"
users_file = "sd254_users.csv"
cards_file = "sd254_cards.csv"

transactions_df = pd.read_csv(os.path.join(origin_data_dir, transactions_file))
transactions_df

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24386895,1999,1,2020,2,27,22:23,$-54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386896,1999,1,2020,2,27,22:24,$54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.0,5541,NaN,No
24386897,1999,1,2020,2,28,07:43,$59.15,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No
24386898,1999,1,2020,2,28,20:10,$43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No


In [3]:
users_df = pd.read_csv(os.path.join(origin_data_dir, users_file))
users_df

,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Jose Faraday,32,70,1987,7,Male,6577 Lexington Lane,9.0,Freeport,NY,11520,40.65,-73.58,$23550,$48010,$87837,703,3
1996,Ximena Richardson,62,65,1957,11,Female,2 Elm Drive,955.0,Independence,KY,41051,38.95,-84.54,$24218,$49378,$104480,740,4
1997,Annika Russell,47,67,1973,1,Female,276 Fifth Boulevard,NaN,Elizabeth,NJ,7201,40.66,-74.19,$15175,$30942,$71066,779,3
1998,Juelz Roman,66,60,1954,2,Male,259 Valley Boulevard,NaN,Camp Hill,PA,17011,40.24,-76.92,$25336,$54654,$27241,618,1


In [4]:
cards_df = pd.read_csv(os.path.join(origin_data_dir, cards_file))
cards_df

,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6141,1997,1,Amex,Credit,300609782832003,01/2024,663,YES,1,$6900,11/2000,2013,No
6142,1997,2,Visa,Credit,4718517475996018,01/2021,492,YES,2,$5700,04/2012,2012,No
6143,1998,0,Mastercard,Credit,5929512204765914,08/2020,237,NO,2,$9200,02/2012,2012,No
6144,1999,0,Mastercard,Debit,5589768928167462,01/2020,630,YES,1,$28074,01/2020,2020,No


# Transactions data preprocessing - 1

#### "Mercaht City" 열이 "ONLINE"인데 "USE Chip"열이 "Chip Transaction"인 경우 "Online Transaction"으로 변경

In [5]:
transactions_df[(transactions_df['Merchant City'] == "ONLINE")]['Use Chip'].unique()

array(['Online Transaction', 'Chip Transaction'], dtype=object)

In [6]:
transactions_df.loc[(transactions_df["Merchant City"] == "ONLINE") & (transactions_df["Use Chip"] == "Chip Transaction"), "Use Chip"] = "Online Transaction"

In [7]:
transactions_df[(transactions_df['Merchant City'] == "ONLINE")]['Use Chip'].unique()

array(['Online Transaction'], dtype=object)

#### Year, Month, Day, Time 열을 합쳐서 하나의 DateTime으로 변경 후 정렬

In [8]:
transactions_df['Datetime'] = pd.to_datetime(
    transactions_df['Year'].astype(str) + '-' + 
    transactions_df['Month'].astype(str) + '-' + 
    transactions_df['Day'].astype(str) + ' ' + 
    transactions_df['Time']
)

In [9]:
transactions_df = transactions_df.drop(columns=["Year", "Month", "Day", "Time"])
cols = ["Datetime"] + [c for c in transactions_df.columns if c != "Datetime"]
transactions_df = transactions_df[cols].sort_values(by="Datetime")
transactions_df

,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
9196264,1991-01-02 07:10:00,791,1,$68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196265,1991-01-02 07:17:00,791,1,$-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196266,1991-01-02 07:21:00,791,1,$113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,No
9196267,1991-01-02 17:30:00,791,1,$114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,No
9196268,1991-01-03 09:03:00,791,1,$251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,No
...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,2020-02-28 23:51:00,1659,2,$7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,No
10296971,2020-02-28 23:53:00,863,1,$49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,No
16828012,2020-02-28 23:56:00,1366,2,$132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,No
15999580,2020-02-28 23:56:00,1300,0,$51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,No


#### "Errors?"의 열 이름을 "Errors"로 변경 후 Error가 여러개일 경우 list로 변환

In [10]:
transactions_df.rename(columns={'Errors?': 'Errors'}, inplace=True)

In [11]:
transactions_df['Errors'] = transactions_df['Errors'].str.split(',')

In [12]:
for trans in transactions_df.iloc:
    if trans['Errors'] is not np.nan:
        print(trans['Errors'])
        print(trans)
        break

['Insufficient Balance']
Datetime             1991-03-14 18:24:00
User                                 791
Card                                   1
Amount                           $150.06
Use Chip               Swipe Transaction
Merchant Name        8108107701014151249
Merchant City                      Burke
Merchant State                        VA
Zip                              22015.0
MCC                                 5300
Errors            [Insufficient Balance]
Is Fraud?                             No
Name: 9196386, dtype: object


#### "Is Fraud?"의 열 이름을 "Fraud"로 변경하고, True/False로 변경

In [13]:
transactions_df.rename(columns={'Is Fraud?': 'Fraud'}, inplace=True)

In [14]:
transactions_df['Fraud'] = transactions_df['Fraud'].map({'No': False, 'Yes': True})

#### "Amount"에서 $ 제거 및 float로 변환

In [15]:
transactions_df['Amount'] = transactions_df['Amount'].str.replace('$', '', regex=False).astype(float)


In [16]:
transactions_df

,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors,Fraud
9196264,1991-01-02 07:10:00,791,1,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196265,1991-01-02 07:17:00,791,1,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196266,1991-01-02 07:21:00,791,1,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196267,1991-01-02 17:30:00,791,1,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,False
9196268,1991-01-03 09:03:00,791,1,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,2020-02-28 23:51:00,1659,2,7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,False
10296971,2020-02-28 23:53:00,863,1,49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,False
16828012,2020-02-28 23:56:00,1366,2,132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,False
15999580,2020-02-28 23:56:00,1300,0,51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,False


In [17]:
transactions_df.dtypes

Datetime          datetime64[ns]
User                       int64
Card                       int64
Amount                   float64
Use Chip                  object
Merchant Name              int64
Merchant City             object
Merchant State            object
Zip                      float64
MCC                        int64
Errors                    object
Fraud                       bool
dtype: object

#### 각 거래에 UUID 할당

In [18]:
import uuid

transactions_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(transactions_df))])
transactions_df

,UUID,Datetime,User,Card,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors,Fraud
9196264,a265d411-31ee-42a2-9270-f115d902e3aa,1991-01-02 07:10:00,791,1,68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196265,7feb55ea-b92c-48d7-907e-372c15b2eff2,1991-01-02 07:17:00,791,1,-68.00,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196266,dcc93159-9074-4525-ab9d-823cc5205e2b,1991-01-02 07:21:00,791,1,113.62,Swipe Transaction,2027553650310142703,Burke,VA,22015.0,5541,NaN,False
9196267,5d4b5a8f-4470-4441-a367-ac24299643da,1991-01-02 17:30:00,791,1,114.73,Swipe Transaction,-7269691894846892021,Burke,VA,22015.0,5411,NaN,False
9196268,b40dc01d-e66d-41a4-8de1-f13c3b506e8f,1991-01-03 09:03:00,791,1,251.71,Swipe Transaction,-3693650930986299431,Burke,VA,22015.0,4814,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20357329,e22a8d9b-5858-4df2-ada9-b3ee510b867d,2020-02-28 23:51:00,1659,2,7.67,Chip Transaction,7231700044622779845,Cosby,TN,37722.0,5300,NaN,False
10296971,e98419c9-ec5e-4373-8b7b-5d5ceceb8379,2020-02-28 23:53:00,863,1,49.06,Chip Transaction,7654254764356253071,North Brunswick,NJ,8902.0,5912,NaN,False
16828012,0e964ea7-1ef0-4bf5-b3d1-d93c53863689,2020-02-28 23:56:00,1366,2,132.73,Chip Transaction,-7398558035733466800,Rockville Centre,NY,11570.0,5812,NaN,False
15999580,e7640cf8-911c-4a61-b99c-39fad04f6594,2020-02-28 23:56:00,1300,0,51.29,Online Transaction,-6458444334611773637,ONLINE,NaN,NaN,4784,NaN,False


# Users data preprocessing

#### UUID 할당

In [19]:
import uuid

users_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(users_df))])
users_df

,UUID,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,100bb8f3-5922-4de9-8500-4ded119e0e53,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,da256a09-1f8a-4109-ab1e-c13f7af4c34a,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,9b937b3b-02f4-44e6-933d-15dc3d9947ac,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,2bd331df-33b4-48ee-8588-f727b099ddd3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,0db0948f-d9f2-47b6-a37d-42f5bb059689,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,45eadb1c-5eb3-4474-8a3f-c1aea3335969,Jose Faraday,32,70,1987,7,Male,6577 Lexington Lane,9.0,Freeport,NY,11520,40.65,-73.58,$23550,$48010,$87837,703,3
1996,9557bf6f-a9f0-4641-9edf-9da0d101da02,Ximena Richardson,62,65,1957,11,Female,2 Elm Drive,955.0,Independence,KY,41051,38.95,-84.54,$24218,$49378,$104480,740,4
1997,4b7ba611-4e15-480e-ae2e-9d5fc935bcc6,Annika Russell,47,67,1973,1,Female,276 Fifth Boulevard,NaN,Elizabeth,NJ,7201,40.66,-74.19,$15175,$30942,$71066,779,3
1998,3fc26c99-51c3-40cc-b62d-8245ddfebec0,Juelz Roman,66,60,1954,2,Male,259 Valley Boulevard,NaN,Camp Hill,PA,17011,40.24,-76.92,$25336,$54654,$27241,618,1


#### "Birth Year" + "Birth Month" ==> "Birth"

In [20]:
users_df['Birth'] = pd.to_datetime(users_df['Birth Year'].astype(str) + '-' + users_df['Birth Month'].astype(str))
birth_idx = users_df.columns.get_loc('Birth Year')
birth_col = users_df.pop('Birth')
users_df.drop(columns=['Birth Year', 'Birth Month'], inplace=True)
users_df.insert(birth_idx, 'Birth', birth_col)

In [21]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,100bb8f3-5922-4de9-8500-4ded119e0e53,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,da256a09-1f8a-4109-ab1e-c13f7af4c34a,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,9b937b3b-02f4-44e6-933d-15dc3d9947ac,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,2bd331df-33b4-48ee-8588-f727b099ddd3,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,0db0948f-d9f2-47b6-a37d-42f5bb059689,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1


#### "Per Capita Income - Zipcode", "Yearly Income - Person", "Total Debt"에서 $ 제거 및 float변환

In [22]:
cols_to_fix = ["Per Capita Income - Zipcode", "Yearly Income - Person", "Total Debt"]
for col in cols_to_fix:
    users_df[col] = users_df[col].replace('[\$,]', '', regex=True).astype(float)

In [23]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,100bb8f3-5922-4de9-8500-4ded119e0e53,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1,da256a09-1f8a-4109-ab1e-c13f7af4c34a,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5
2,9b937b3b-02f4-44e6-933d-15dc3d9947ac,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5
3,2bd331df-33b4-48ee-8588-f727b099ddd3,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4
4,0db0948f-d9f2-47b6-a37d-42f5bb059689,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [24]:
users_df.dtypes

UUID                                   object
Person                                 object
Current Age                             int64
Retirement Age                          int64
Birth                          datetime64[ns]
Gender                                 object
Address                                object
Apartment                             float64
City                                   object
State                                  object
Zipcode                                 int64
Latitude                              float64
Longitude                             float64
Per Capita Income - Zipcode           float64
Yearly Income - Person                float64
Total Debt                            float64
FICO Score                              int64
Num Credit Cards                        int64
dtype: object

# Cards data preprocessing

#### 각 카드별 UUID 할당

In [25]:
import uuid

cards_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(cards_df))])

#### "User"에 index대신 users_df의 UUID로 변환

In [26]:
cards_df['User'] = cards_df['User'].map(users_df['UUID'])

#### "Expires", "Acct Open Date"를 datetime으로 변경

In [27]:
from _codecs import encode
# for col in [, "Acct Open Date"]:
cards_df["Expires"] = pd.to_datetime(cards_df["Expires"], errors='coerce', format='mixed') + pd.offsets.MonthEnd(0)
cards_df["Acct Open Date"] = pd.to_datetime(cards_df["Acct Open Date"], errors='coerce', format='mixed')

#### "Credit Limit"에 $ 제거 및 float형 변환

In [28]:
cards_df['Credit Limit'] = cards_df['Credit Limit'].replace('[\$,]', '', regex=True).astype(float)

#### "Has Chip", "Card on Dark Web"을 bool type으로 변환

In [29]:
cards_df['Has Chip'] = cards_df['Has Chip'].map({'NO': False, 'YES': True})
cards_df["Card on Dark Web"] = cards_df["Card on Dark Web"].map({'No': False, 'Yes': True})

In [30]:
cards_df

,UUID,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,b9d4ec69-f505-4df5-882f-e0b018c57128,100bb8f3-5922-4de9-8500-4ded119e0e53,0,Visa,Debit,4344676511950444,2022-12-31,623,True,2,24295.0,2002-09-01,2008,False
1,1dc000a1-f9e8-4015-b88e-db9e84ae3f63,100bb8f3-5922-4de9-8500-4ded119e0e53,1,Visa,Debit,4956965974959986,2020-12-31,393,True,2,21968.0,2014-04-01,2014,False
2,04448111-defd-4122-92f0-13f700823a85,100bb8f3-5922-4de9-8500-4ded119e0e53,2,Visa,Debit,4582313478255491,2024-02-29,719,True,2,46414.0,2003-07-01,2004,False
3,5db93bec-c09e-4965-98b6-696041984a61,100bb8f3-5922-4de9-8500-4ded119e0e53,3,Visa,Credit,4879494103069057,2024-08-31,693,False,1,12400.0,2003-01-01,2012,False
4,a5ade466-dbf7-41dd-98d8-de3d80617d5b,100bb8f3-5922-4de9-8500-4ded119e0e53,4,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31,75,True,1,28.0,2008-09-01,2009,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6141,88675cae-833c-4358-8d0d-7a26d00ba411,4b7ba611-4e15-480e-ae2e-9d5fc935bcc6,1,Amex,Credit,300609782832003,2024-01-31,663,True,1,6900.0,2000-11-01,2013,False
6142,78cacdbe-3453-49f6-ba78-608aec2ebd51,4b7ba611-4e15-480e-ae2e-9d5fc935bcc6,2,Visa,Credit,4718517475996018,2021-01-31,492,True,2,5700.0,2012-04-01,2012,False
6143,fbf185c9-9fc1-45ff-9d77-f270356124c7,3fc26c99-51c3-40cc-b62d-8245ddfebec0,0,Mastercard,Credit,5929512204765914,2020-08-31,237,False,2,9200.0,2012-02-01,2012,False
6144,baf59f64-5b64-4b5c-aee6-8f9e0a05e981,e447a189-8c7e-4ecd-a327-232396255992,0,Mastercard,Debit,5589768928167462,2020-01-31,630,True,1,28074.0,2020-01-01,2020,False


In [31]:
cards_df.dtypes

UUID                             object
User                             object
CARD INDEX                        int64
Card Brand                       object
Card Type                        object
Card Number                       int64
Expires                  datetime64[ns]
CVV                               int64
Has Chip                           bool
Cards Issued                      int64
Credit Limit                    float64
Acct Open Date           datetime64[ns]
Year PIN last Changed             int64
Card on Dark Web                   bool
dtype: object

# Users data preprocessing - 2

#### users_df에 각 user의 card의 UUID 추가

In [32]:
users_df['Cards'] = users_df['UUID'].map(cards_df.groupby('User')['UUID'].apply(list)).apply(lambda x: x if isinstance(x, list) else [])

In [33]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,Cards
0,100bb8f3-5922-4de9-8500-4ded119e0e53,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5,"[b9d4ec69-f505-4df5-882f-e0b018c57128, 1dc000a..."
1,da256a09-1f8a-4109-ab1e-c13f7af4c34a,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5,"[6597d309-7419-4372-abd3-e18a179351b3, 9bafe23..."
2,9b937b3b-02f4-44e6-933d-15dc3d9947ac,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5,"[5786dcb9-a2e8-4a84-b8fe-25b5706854c1, 40a8d07..."
3,2bd331df-33b4-48ee-8588-f727b099ddd3,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4,"[bbb4d680-fe1c-458b-b0ca-a106c1f8ee69, 64d14dc..."
4,0db0948f-d9f2-47b6-a37d-42f5bb059689,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1,[7d17db4b-3296-4461-8ae9-34f6f1dfb939]


# Transactions data preprocessing - 2

#### "User", "Card" 열을 각각 users_df의 UUID, cards_df의 UUID로 변경

In [34]:
transactions_df['User'] = transactions_df['User'].map(users_df['UUID'])

In [35]:
transactions_df = transactions_df.merge(
    cards_df[['User', 'CARD INDEX', 'UUID']].rename(columns={'UUID': 'UUID_card'}), 
    left_on=['User', 'Card'], 
    right_on=['User', 'CARD INDEX'], 
    how='left'
)
transactions_df['Card'] = transactions_df['UUID_card']
transactions_df.drop(columns=['CARD INDEX', 'UUID_card'], inplace=True)

#### merchants_df 분리

In [36]:
merchants_df = transactions_df[['Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC']].copy()

In [37]:
import uuid

merchants_df = merchants_df.drop_duplicates().reset_index(drop=True)
merchants_df.insert(0, 'UUID', [str(uuid.uuid4()) for _ in range(len(merchants_df))])

merchants_df

,UUID,Merchant Name,Merchant City,Merchant State,Zip,MCC
0,71831437-5936-45fa-909f-9701bae937de,2027553650310142703,Burke,VA,22015.0,5541
1,c1bd08b8-95db-4a1b-833c-a16a9eea1962,-7269691894846892021,Burke,VA,22015.0,5411
2,4ba09010-581b-4c47-addb-8d36d79cb772,-3693650930986299431,Burke,VA,22015.0,4814
3,d3d84660-0175-4608-a0a8-d86def4bba29,3189517333335617109,Fairfax,VA,22030.0,5311
4,f032a8d5-c061-4067-8dbd-48de5d43b102,5701841789931834110,Burke,VA,22015.0,5411
...,...,...,...,...,...,...
283976,81a9eb00-c9be-4fcf-8398-a0e6906ffb56,-7911106634782211204,Portland,ME,4103.0,7349
283977,1e6e6ef1-a54a-4aa3-8e3e-108442ec826e,-8893969934710155907,Perris,CA,92570.0,7538
283978,9df0d868-c8d5-4d2b-b11b-a5f99f391d03,-2248444526575079323,Phoenix,AZ,85043.0,5655
283979,2a479725-19cd-4f59-a4ae-8b8fa13452b3,-7664872663026589029,Bronx,NY,10455.0,8021


#### transactions_df의 "Merchant Name" 자리에 merchants_df의 "UUID"로 삽입

In [38]:
# transactions_df와 병합
transactions_df = transactions_df.merge(
    merchants_df.rename(columns={'UUID': 'Merchant'}), 
    on=['Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC'], 
    how='left'
)

# 'Merchant' 컬럼을 'Merchant Name' 위치로 이동
cols = transactions_df.columns.tolist()
merchant_name_idx = cols.index('Merchant Name')
cols.insert(merchant_name_idx, cols.pop(cols.index('Merchant')))
transactions_df = transactions_df[cols]

#### transactions_df에서 "Merchant"만 놔두고, "Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC"는 drop

In [39]:
transactions_df.drop(columns=["Merchant Name", "Merchant City", "Merchant State", "Zip", "MCC"], inplace=True)

In [40]:
transactions_df

,UUID,Datetime,User,Card,Amount,Use Chip,Merchant,Errors,Fraud
0,a265d411-31ee-42a2-9270-f115d902e3aa,1991-01-02 07:10:00,1cacc056-29b2-44bc-861f-b18cae70ced0,f3029bc4-9bbe-4d47-8596-84a56a487d26,68.00,Swipe Transaction,71831437-5936-45fa-909f-9701bae937de,NaN,False
1,7feb55ea-b92c-48d7-907e-372c15b2eff2,1991-01-02 07:17:00,1cacc056-29b2-44bc-861f-b18cae70ced0,f3029bc4-9bbe-4d47-8596-84a56a487d26,-68.00,Swipe Transaction,71831437-5936-45fa-909f-9701bae937de,NaN,False
2,dcc93159-9074-4525-ab9d-823cc5205e2b,1991-01-02 07:21:00,1cacc056-29b2-44bc-861f-b18cae70ced0,f3029bc4-9bbe-4d47-8596-84a56a487d26,113.62,Swipe Transaction,71831437-5936-45fa-909f-9701bae937de,NaN,False
3,5d4b5a8f-4470-4441-a367-ac24299643da,1991-01-02 17:30:00,1cacc056-29b2-44bc-861f-b18cae70ced0,f3029bc4-9bbe-4d47-8596-84a56a487d26,114.73,Swipe Transaction,c1bd08b8-95db-4a1b-833c-a16a9eea1962,NaN,False
4,b40dc01d-e66d-41a4-8de1-f13c3b506e8f,1991-01-03 09:03:00,1cacc056-29b2-44bc-861f-b18cae70ced0,f3029bc4-9bbe-4d47-8596-84a56a487d26,251.71,Swipe Transaction,4ba09010-581b-4c47-addb-8d36d79cb772,NaN,False
...,...,...,...,...,...,...,...,...,...
24386895,e22a8d9b-5858-4df2-ada9-b3ee510b867d,2020-02-28 23:51:00,3dd2db8f-3f9c-416a-a999-b12ebb560419,52690d54-f11e-4dfc-977a-269ee58bc9ba,7.67,Chip Transaction,7f9d069b-fb31-4007-be1f-f8d75b34cf72,NaN,False
24386896,e98419c9-ec5e-4373-8b7b-5d5ceceb8379,2020-02-28 23:53:00,b556c58a-bb0a-42ce-a7b1-22f428d2f179,3848f2d0-3634-4f4f-8653-03e2bba80764,49.06,Chip Transaction,2e5b83cf-9829-4a45-a552-1fb1fc66a9da,NaN,False
24386897,0e964ea7-1ef0-4bf5-b3d1-d93c53863689,2020-02-28 23:56:00,b82ebe10-a33c-4399-adf0-d64c21b6cfe6,a3592b1b-6682-43e4-a9bd-ae19c39d9048,132.73,Chip Transaction,90d763f5-6cc7-4ddd-ab03-5a8479210bb3,NaN,False
24386898,e7640cf8-911c-4a61-b99c-39fad04f6594,2020-02-28 23:56:00,2a053079-a999-4ec5-8bda-3b6a7ccc05c9,1e0f94bf-26ad-47c8-8cb7-ef417e344aca,51.29,Online Transaction,74680c55-20ce-4435-a345-8b55d951cdce,NaN,False


# Merchants data preprocessing

#### column명 수정

In [41]:
merchants_df = merchants_df.rename(columns={'Merchant Name': 'Name', 'Merchant City': 'City', 'Merchant State': 'State', 'Zip':'Zipcode'})

In [42]:
merchants_df

,UUID,Name,City,State,Zipcode,MCC
0,71831437-5936-45fa-909f-9701bae937de,2027553650310142703,Burke,VA,22015.0,5541
1,c1bd08b8-95db-4a1b-833c-a16a9eea1962,-7269691894846892021,Burke,VA,22015.0,5411
2,4ba09010-581b-4c47-addb-8d36d79cb772,-3693650930986299431,Burke,VA,22015.0,4814
3,d3d84660-0175-4608-a0a8-d86def4bba29,3189517333335617109,Fairfax,VA,22030.0,5311
4,f032a8d5-c061-4067-8dbd-48de5d43b102,5701841789931834110,Burke,VA,22015.0,5411
...,...,...,...,...,...,...
283976,81a9eb00-c9be-4fcf-8398-a0e6906ffb56,-7911106634782211204,Portland,ME,4103.0,7349
283977,1e6e6ef1-a54a-4aa3-8e3e-108442ec826e,-8893969934710155907,Perris,CA,92570.0,7538
283978,9df0d868-c8d5-4d2b-b11b-a5f99f391d03,-2248444526575079323,Phoenix,AZ,85043.0,5655
283979,2a479725-19cd-4f59-a4ae-8b8fa13452b3,-7664872663026589029,Bronx,NY,10455.0,8021


In [43]:
merchants_df.dtypes

UUID        object
Name         int64
City        object
State       object
Zipcode    float64
MCC          int64
dtype: object

In [44]:
users_df.head()

,UUID,Person,Current Age,Retirement Age,Birth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,Cards
0,100bb8f3-5922-4de9-8500-4ded119e0e53,Hazel Robinson,53,66,1966-11-01,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.0,59696.0,127613.0,787,5,"[b9d4ec69-f505-4df5-882f-e0b018c57128, 1dc000a..."
1,da256a09-1f8a-4109-ab1e-c13f7af4c34a,Sasha Sadr,53,68,1966-12-01,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,37891.0,77254.0,191349.0,701,5,"[6597d309-7419-4372-abd3-e18a179351b3, 9bafe23..."
2,9b937b3b-02f4-44e6-933d-15dc3d9947ac,Saanvi Lee,81,67,1938-11-01,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,22681.0,33483.0,196.0,698,5,"[5786dcb9-a2e8-4a84-b8fe-25b5706854c1, 40a8d07..."
3,2bd331df-33b4-48ee-8588-f727b099ddd3,Everlee Clark,63,63,1957-01-01,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,163145.0,249925.0,202328.0,722,4,"[bbb4d680-fe1c-458b-b0ca-a106c1f8ee69, 64d14dc..."
4,0db0948f-d9f2-47b6-a37d-42f5bb059689,Kyle Peterson,43,70,1976-09-01,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,53797.0,109687.0,183855.0,675,1,[7d17db4b-3296-4461-8ae9-34f6f1dfb939]


In [45]:
cards_df.head()

,UUID,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,b9d4ec69-f505-4df5-882f-e0b018c57128,100bb8f3-5922-4de9-8500-4ded119e0e53,0,Visa,Debit,4344676511950444,2022-12-31,623,True,2,24295.0,2002-09-01,2008,False
1,1dc000a1-f9e8-4015-b88e-db9e84ae3f63,100bb8f3-5922-4de9-8500-4ded119e0e53,1,Visa,Debit,4956965974959986,2020-12-31,393,True,2,21968.0,2014-04-01,2014,False
2,04448111-defd-4122-92f0-13f700823a85,100bb8f3-5922-4de9-8500-4ded119e0e53,2,Visa,Debit,4582313478255491,2024-02-29,719,True,2,46414.0,2003-07-01,2004,False
3,5db93bec-c09e-4965-98b6-696041984a61,100bb8f3-5922-4de9-8500-4ded119e0e53,3,Visa,Credit,4879494103069057,2024-08-31,693,False,1,12400.0,2003-01-01,2012,False
4,a5ade466-dbf7-41dd-98d8-de3d80617d5b,100bb8f3-5922-4de9-8500-4ded119e0e53,4,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31,75,True,1,28.0,2008-09-01,2009,False


# Save processed data

In [46]:
import os

result_dir = "data/processed_data"
os.makedirs(result_dir, exist_ok=True)

transactions_df.to_csv(os.path.join(result_dir, "transactions.csv"), index=False)
cards_df.to_csv(os.path.join(result_dir, "cards.csv"), index=False)
users_df.to_csv(os.path.join(result_dir, "users.csv"), index=False)
merchants_df.to_csv(os.path.join(result_dir, "merchants.csv"), index=False)